# Credit Card Fraud Analytics — Exploratory Data Analysis

## Phase 3: Exploratory Data Analysis (EDA)



The goal of this phase is to discover patterns of normal and fraudulent transactions using real data.

**Focus areas**

- Class Imbalance
- Transaction Amount
- Fraud vs Normal Amount
- Amount Bands
- Time-of-Day
- V1–V28 PCA Features
- Correlation with Fraud
- Business KPIs

Note: V1 through V28 are anonymous PCA features; we do not assume any business meaning for them.


## 1. Import Libraries


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")


## 2. Load Clean Dataset


In [ ]:
DATA_PATH = "../data/creditcard_clean.csv"
df = pd.read_csv(DATA_PATH)
print(f"Dataset shape: {df.shape}")
display(df.head())


## 3. Basic Overview


In [ ]:
overview = pd.DataFrame({
    "metric": ["Rows","Columns","Fraud transactions","Normal transactions","Fraud rate (%)"],
    "value": [len(df), df.shape[1], int(df["Class"].sum()),
              int((df["Class"] == 0).sum()), df["Class"].mean()*100]
})
display(overview)


## 4. Class Distribution


In [ ]:
class_counts = df["Class"].value_counts().sort_index()
class_summary = pd.DataFrame({
    "Class": ["Normal", "Fraud"],
    "Count": [class_counts.get(0,0), class_counts.get(1,0)]
})
class_summary["Percentage"] = class_summary["Count"] / len(df) * 100
display(class_summary)


In [ ]:
plt.figure(figsize=(7,5))
plt.bar(class_summary["Class"], class_summary["Count"])
plt.title("Transaction Class Distribution")
plt.xlabel("Transaction Class")
plt.ylabel("Number of Transactions")
plt.tight_layout()
plt.show()


**Interpretation:** The severe class imbalance in later stages makes Accuracy an inappropriate primary metric.


## 5. Transaction Amount — Descriptive Statistics


In [ ]:
amount_summary = df["Amount"].describe().to_frame("Amount")
amount_summary.loc["median"] = df["Amount"].median()
amount_summary.loc["zero_amount_count"] = (df["Amount"] == 0).sum()
amount_summary.loc["zero_amount_percentage"] = (df["Amount"] == 0).mean()*100
display(amount_summary)


## 6. Overall Amount Distribution


In [ ]:
plt.figure(figsize=(10,5))
plt.hist(df["Amount"], bins=100)
plt.title("Overall Transaction Amount Distribution")
plt.xlabel("Transaction Amount")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(10,5))
plt.hist(np.log1p(df["Amount"]), bins=100)
plt.title("Log-Transformed Transaction Amount Distribution")
plt.xlabel("log(1 + Amount)")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()


## 7. Amount Distribution by Class


In [ ]:
normal_amount = df.loc[df["Class"] == 0, "Amount"]
fraud_amount = df.loc[df["Class"] == 1, "Amount"]

amount_by_class = pd.DataFrame({
    "Normal": normal_amount.describe(),
    "Fraud": fraud_amount.describe()
})
amount_by_class.loc["median"] = [normal_amount.median(), fraud_amount.median()]
display(amount_by_class)


In [ ]:
plt.figure(figsize=(9,5))
plt.boxplot([normal_amount, fraud_amount],
            labels=["Normal","Fraud"], showfliers=False)
plt.title("Transaction Amount by Class")
plt.xlabel("Transaction Class")
plt.ylabel("Transaction Amount")
plt.tight_layout()
plt.show()


Compare Mean and Median together; their difference can indicate skewness and the presence of extreme values.

## 8. Fraud Rate Across Amount Bands


In [ ]:
amount_bins = [-0.01,10,25,50,100,250,500,1000,2500,5000,np.inf]
amount_labels = ["0–10","10–25","25–50","50–100","100–250",
                 "250–500","500–1,000","1,000–2,500","2,500–5,000","5,000+"]

df["AmountBand"] = pd.cut(df["Amount"], bins=amount_bins, labels=amount_labels)

amount_band_summary = (
    df.groupby("AmountBand", observed=False)
      .agg(transactions=("Class","size"),
           fraud_count=("Class","sum"),
           fraud_rate=("Class","mean"))
)
amount_band_summary["fraud_rate"] *= 100
display(amount_band_summary)


In [ ]:
plot_data = amount_band_summary.reset_index()
plt.figure(figsize=(12,5))
plt.bar(plot_data["AmountBand"].astype(str), plot_data["fraud_rate"])
plt.title("Fraud Rate by Transaction Amount Band")
plt.xlabel("Amount Band")
plt.ylabel("Fraud Rate (%)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 9. Prepare Time Features


In [ ]:
df["TimeHours"] = df["Time"] / 3600
df["HourOfDay"] = (df["Time"] // 3600) % 24
display(df[["Time","TimeHours","HourOfDay"]].head())


This dataset covers approximately two days; therefore, we do not use it to make claims about monthly or seasonal trends.

## 10. Transaction Volume by Hour


In [ ]:
hourly_volume = (
    df.groupby("HourOfDay")
      .agg(transactions=("Class","size"),
           fraud_count=("Class","sum"),
           fraud_rate=("Class","mean"))
)
hourly_volume["fraud_rate"] *= 100
display(hourly_volume)


In [ ]:
plt.figure(figsize=(12,5))
plt.bar(hourly_volume.index, hourly_volume["transactions"])
plt.title("Transaction Volume by Hour of Day")
plt.xlabel("Hour of Day")
plt.ylabel("Number of Transactions")
plt.xticks(range(24))
plt.tight_layout()
plt.show()


## 11. Fraud Rate by Hour


In [ ]:
plt.figure(figsize=(12,5))
plt.bar(hourly_volume.index, hourly_volume["fraud_rate"])
plt.title("Fraud Rate by Hour of Day")
plt.xlabel("Hour of Day")
plt.ylabel("Fraud Rate (%)")
plt.xticks(range(24))
plt.tight_layout()
plt.show()


For hours with a high Fraud Rate, also check the transaction count; a high rate on a small sample can be unstable.

## 12. Fraud vs Normal Amount Quantiles


In [ ]:
quantiles = [0.25,0.50,0.75,0.90,0.95,0.99]
amount_quantiles = pd.DataFrame({
    "Normal": normal_amount.quantile(quantiles),
    "Fraud": fraud_amount.quantile(quantiles)
})
amount_quantiles.index = [f"{int(q*100)}th_percentile" for q in quantiles]
display(amount_quantiles)


## 13. V1–V28 Distribution Summary


In [ ]:
v_features = [f"V{i}" for i in range(1,29)]
v_summary = df[v_features].describe().T
v_summary["missing"] = df[v_features].isna().sum()
v_summary["unique"] = df[v_features].nunique()
display(v_summary)


## 14. V1–V28 Mean by Class


In [ ]:
v_means_by_class = df.groupby("Class")[v_features].mean().T
v_means_by_class.columns = ["Normal_mean","Fraud_mean"]
v_means_by_class["absolute_mean_difference"] = (
    v_means_by_class["Fraud_mean"] - v_means_by_class["Normal_mean"]
).abs()
v_means_by_class = v_means_by_class.sort_values(
    "absolute_mean_difference", ascending=False
)
display(v_means_by_class)


These differences are only exploratory signals and do not, by themselves, prove predictive power.

## 15. Correlation with Fraud Target


In [ ]:
correlations = (
    df[v_features + ["Amount","Class"]]
      .corr(numeric_only=True)["Class"]
      .drop("Class")
      .sort_values(key=np.abs, ascending=False)
)
display(correlations.to_frame("correlation_with_class"))


Correlation is a univariate measure, and a low correlation does not necessarily mean that a feature is useless.

## 16. Top Features by Absolute Correlation


In [ ]:
top_correlations = correlations.head(10).sort_values()
plt.figure(figsize=(9,6))
plt.barh(top_correlations.index, top_correlations.values)
plt.title("Top Features by Absolute Correlation with Fraud")
plt.xlabel("Correlation with Class")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()


## 17. Selected Correlation Matrix


In [ ]:
selected_features = ["Time","Amount"] + v_features[:8] + ["Class"]
corr_matrix = df[selected_features].corr()
display(corr_matrix)


## 18. Fraud Transaction Share vs Fraud Amount Share


In [ ]:
fraud_transaction_share = df["Class"].mean() * 100
fraud_amount_share = (
    df.loc[df["Class"] == 1, "Amount"].sum()
    / df["Amount"].sum() * 100
)

share_summary = pd.DataFrame({
    "metric": ["Fraud transaction share","Fraud amount share"],
    "percentage": [fraud_transaction_share, fraud_amount_share]
})
display(share_summary)


This KPI shows the difference between the number of fraudulent transactions and the financial value of fraud, and it is useful for Power BI.

## 19. EDA KPI Summary


In [ ]:
eda_kpis = pd.DataFrame({
    "KPI": [
        "Total transactions","Fraud transactions","Fraud rate (%)",
        "Total transaction amount","Fraud transaction amount",
        "Fraud amount share (%)","Median normal amount",
        "Median fraud amount","Mean normal amount","Mean fraud amount"
    ],
    "Value": [
        len(df), int(df["Class"].sum()), df["Class"].mean()*100,
        df["Amount"].sum(), df.loc[df["Class"] == 1,"Amount"].sum(),
        fraud_amount_share, normal_amount.median(), fraud_amount.median(),
        normal_amount.mean(), fraud_amount.mean()
    ]
})
display(eda_kpis)


## 20. Save EDA Outputs


In [ ]:
OUTPUT_DIR = "../data"

amount_band_summary.to_csv(f"{OUTPUT_DIR}/eda_amount_band_summary.csv")
hourly_volume.to_csv(f"{OUTPUT_DIR}/eda_hourly_summary.csv")
v_means_by_class.to_csv(f"{OUTPUT_DIR}/eda_v_feature_class_means.csv")
correlations.to_frame("correlation_with_class").to_csv(
    f"{OUTPUT_DIR}/eda_feature_correlations.csv"
)
eda_kpis.to_csv(f"{OUTPUT_DIR}/eda_kpis.csv", index=False)

print("EDA output tables saved successfully.")


## 21. Phase 3 Conclusions

After running the Notebook, answer the following questions based on the actual output:

1. How severe is the class imbalance?
2. How does Amount differ between Fraud and Normal transactions?
3. Which Amount Band has the highest Fraud Rate?
4. Does Fraud Rate vary significantly across different hours of the day?
5. Which V features show the strongest difference or correlation with Fraud?
6. How does the share of Fraud in terms of transaction count compare to its share in terms of total amount?

### Next Phase

**Phase 4 — Fraud Analysis**

In the next phase, we will focus more deeply on Fraud: segmentation, statistical comparison, fraud KPIs, and risk patterns.